> 在线讲义：[第 19 章 · 评测：97.1% 和 96.4%，到底谁更强](https://wikieden.github.io/vla-from-scratch/lessons/19-evaluation.html) · 本书《VLA 从零到 π0》

# 第 19 章 · 评测：97.1% 和 96.4%，到底谁更强

配套讲义：仓库里 `vla-deep-dive/notes/lessons/19-evaluation.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `LIBERO` | 4 × 10 × 50 = 2000 回合 |
| 第 2 步 | `wilson_ci(k, n)` | 7/10 → [0.397, 0.892]；57/80 → [0.605, 0.800]；1942/2000 → [0.963, 0.977] |
| 第 3 步 | `two_prop_z(k1, n1, k2, n2)` | z = 1.25、p = 0.21；z = 1.82、p = 0.07 |
| 第 3 步 | `OFT_PROTOCOLS` | 95.3 / 94.5 / 97.1 |

数据取自 openvla-oft-2502.19645.txt 表 I、openvla-2406.09246.txt 表 2、FastWAM README 复现表。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
LIBERO = {"suites": 4, "tasks_per_suite": 10, "episodes_per_task": 50}
OFT_PROTOCOLS = {"① 过滤数据、只有第三人称图": 95.3, "② 原始数据、加腕部图和状态": 94.5, "③ 过滤数据、加腕部图和状态": 97.1}

**函数 `wilson_ci`**：成功 k 次、共 n 回合的 Wilson 区间。

In [ ]:
def wilson_ci(k, n, z=1.96):
    """成功 k 次、共 n 回合的 Wilson 区间。"""
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

**函数 `two_prop_z`**：两比例 z 检验（合并成功率），返回 (z, 双侧 p 值)。

In [ ]:
def two_prop_z(k1, n1, k2, n2):
    """两比例 z 检验（合并成功率），返回 (z, 双侧 p 值)。"""
    p1, p2, pp = k1 / n1, k2 / n2, (k1 + k2) / (n1 + n2)
    z = (p1 - p2) / math.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
    return z, math.erfc(abs(z) / math.sqrt(2))

**函数 `successes`**：论文只给百分比时，反推成功次数。

In [ ]:
def successes(rate_pct, n):
    """论文只给百分比时，反推成功次数。"""
    return round(rate_pct / 100 * n)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 LIBERO 一次评测多少回合")
total = LIBERO["suites"] * LIBERO["tasks_per_suite"] * LIBERO["episodes_per_task"]
print(f"  4 套件 × 10 任务 × 50 回合 = {total}")

In [ ]:
print("\n第二步 Wilson 区间")
for k, n in ((7, 10), (successes(71.3, 80), 80), (successes(97.1, 2000), 2000)):
    lo, hi = wilson_ci(k, n)
    print(f"  {k}/{n}：[{lo:.3f}, {hi:.3f}]，宽 {100 * (hi - lo):.1f} 个点")

In [ ]:
print("\n第三步 两比例 z 检验")
for name, (a, b, n) in {"OFT vs FastWAM": (97.1, 96.4, 2000), "OpenVLA bf16 vs int8": (71.3, 58.1, 80)}.items():
    z, p = two_prop_z(successes(a, n), n, successes(b, n), n)
    print(f"  {name}（各 {n} 回合）：z = {z:.2f}，p = {p:.2f}")
for k, v in OFT_PROTOCOLS.items():
    print(f"  OFT 协议 {k}：{v}%")

## 3. 代码里才看得到的

1. 论文只给百分比时，代码按 $k = \mathrm{round}(\hat p \cdot n)$ 反推成功次数：97.1% × 2000 = 1942、71.3% × 80 = 57.04 → 57；所以算出的区间和论文写法会差在第三位小数。
2. `two_prop_z` 的 p 值用 `math.erfc(|z|/√2)`，就是标准正态的双侧尾概率，不用装 scipy。
3. OFT 论文没写每个套件评测了多少回合；代码按 LIBERO 常用的每任务 50 回合（和 FastWAM 复现表一致）估算，这是假设。

## 4. 自己改一改

- 把 `wilson_ci` 的 z 换成 2.576（99%），看区间宽了多少。
- 用 `two_prop_z` 找出"97.1% vs 96.4%"要各测多少回合才显著（p < 0.05）。
- 给 `wilson_ci` 喂 0 / 10 和 10 / 10，比较和 $\hat p \pm 1.96\,\mathrm{SE}$ 的区别。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 电脑题考千次分数稳，上真路考几次误差大；
> 十投七中，真实在四成到九成之间；
> 差距跟标准误比，比之前看是不是同一张卷子；
> 都考满分就换难卷、比速度。